# 05 — Energy Demand Forecasting with TensorFlow

**Goal:** forecast the next 14 days of German electricity consumption for capacity planning, compare a TensorFlow LSTM with honest seasonal baselines, quantify uncertainty, and package a tested model artifact.

[Open in Google Colab](https://colab.research.google.com/github/Jorgoluka100/uni_projects/blob/main/05_Energy_Demand_Forecasting_with_TensorFlow.ipynb)

This notebook uses real public data, preserves chronological order, fits preprocessing on training data only, evaluates once on an untouched test period, and generates every reported metric during execution.

## Business framing

Grid and operations teams need forward demand estimates to schedule supply, maintenance and reserve capacity. The prediction target is daily electricity consumption for the next 14 days. Success means beating both a last-value baseline and a 7-day seasonal baseline on test MAE. Forecast intervals are included for planning—not as guaranteed bounds.

In [1]:
import json, os, random, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
import tensorflow as tf

SEED=42
os.environ['PYTHONHASHSEED']=str(SEED); random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
warnings.filterwarnings('ignore'); sns.set_theme(style='whitegrid')
ROOT=Path('/content/energy_forecast' if Path('/content').exists() else './energy_forecast'); ROOT.mkdir(parents=True,exist_ok=True)
print({'tensorflow':tf.__version__,'seed':SEED,'devices':[d.device_type for d in tf.config.list_physical_devices()]})

{'tensorflow': '2.20.0', 'seed': 42, 'devices': ['CPU']}


## 1. Real dataset and data contract

Source: [Open Power System Data time-series package](https://data.open-power-system-data.org/time_series/) mirrored in the project-maintainer repository. It contains German daily electricity consumption, wind and solar generation from 2006–2017. This project uses `Consumption` as the target and weather-linked generation columns only as historical context where available. Review the upstream data package and licence before commercial reuse.

In [2]:
DATA_URL='https://raw.githubusercontent.com/jenfly/opsd/master/opsd_germany_daily.csv'
df=pd.read_csv(DATA_URL,parse_dates=['Date']).sort_values('Date').set_index('Date')
df=df.rename(columns={'Consumption':'consumption','Wind':'wind','Solar':'solar','Wind+Solar':'wind_solar'})
required=['consumption']; assert set(required)<=set(df.columns)
assert df.index.is_monotonic_increasing and df.index.is_unique
assert (df['consumption']>0).all()
expected=pd.date_range(df.index.min(),df.index.max(),freq='D')
missing_dates=expected.difference(df.index)
print({'rows':len(df),'start':str(df.index.min().date()),'end':str(df.index.max().date()),'missing_dates':len(missing_dates),'missing_target':int(df.consumption.isna().sum())})
display(df.head()); display(df.describe().T)
assert len(df)==4383 and len(missing_dates)==0 and df.consumption.notna().all()

{'rows': 4383, 'start': '2006-01-01', 'end': '2017-12-31', 'missing_dates': 0, 'missing_target': 0}
            consumption  wind  solar  wind_solar
Date                                            
2006-01-01     1069.184   NaN    NaN         NaN
2006-01-02     1380.521   NaN    NaN         NaN
2006-01-03     1442.533   NaN    NaN         NaN
2006-01-04     1457.217   NaN    NaN         NaN
2006-01-05     1477.131   NaN    NaN         NaN
              count         mean         std  ...       50%         75%       max
consumption  4383.0  1338.675836  165.775710  ...  1367.123  1457.76100  1709.568
wind         2920.0   164.814173  143.692732  ...   119.098   217.90025   826.278
solar        2188.0    89.258695   58.550099  ...    86.407   135.07150   241.580
wind_solar   2187.0   272.663481  146.319884  ...   240.991   338.98800   851.556

[4 rows x 8 columns]


## 2. Exploration without future leakage

In [3]:
fig,axs=plt.subplots(2,1,figsize=(13,8))
df.consumption.plot(ax=axs[0],color='#2457C5',lw=.8,title='German daily electricity consumption')
df.consumption.loc['2016':].plot(ax=axs[1],color='#E36A2E',lw=1,title='Recent two-year view')
for ax in axs: ax.set_ylabel('GWh'); ax.set_xlabel('');
plt.tight_layout(); plt.show()
calendar=pd.DataFrame({'consumption':df.consumption,'weekday':df.index.day_name(),'month':df.index.month})
fig,axs=plt.subplots(1,2,figsize=(13,4)); sns.boxplot(data=calendar,x='weekday',y='consumption',order=['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday'],ax=axs[0]); axs[0].tick_params(axis='x',rotation=35); sns.boxplot(data=calendar,x='month',y='consumption',ax=axs[1]); plt.tight_layout(); plt.show()

## 3. Chronological split and leakage-safe windows

The first 70% trains the model, the next 15% selects training duration and the final 15% is untouched until evaluation. The scaler is fitted only on the training period. Each sample uses the previous 60 days to predict the next 14 days.

In [4]:
LOOKBACK,HORIZON=60,14
n=len(df); train_end=int(.70*n); val_end=int(.85*n)
values=df[['consumption']].to_numpy('float32')
scaler=StandardScaler().fit(values[:train_end]); scaled=scaler.transform(values).astype('float32')

def make_windows(first_target,last_target):
    X,y,dates=[],[],[]
    for i in range(max(first_target,LOOKBACK),last_target-HORIZON+1):
        X.append(scaled[i-LOOKBACK:i]); y.append(scaled[i:i+HORIZON,0]); dates.append(df.index[i])
    return np.asarray(X),np.asarray(y),np.asarray(dates)
X_train,y_train,d_train=make_windows(0,train_end)
X_val,y_val,d_val=make_windows(train_end,val_end)
X_test,y_test,d_test=make_windows(val_end,n)
assert d_train.max()<d_val.min()<d_test.min()
print({'train':X_train.shape,'validation':X_val.shape,'test':X_test.shape,'train_last_target_start':str(pd.Timestamp(d_train.max()).date()),'test_first_target_start':str(pd.Timestamp(d_test.min()).date())})

{'train': (2995, 60, 1), 'validation': (644, 60, 1), 'test': (645, 60, 1), 'train_last_target_start': '2014-05-13', 'test_first_target_start': '2016-03-14'}


## 4. Baselines that the neural network must beat

In [5]:
def invert(x): return scaler.inverse_transform(np.asarray(x).reshape(-1,1)).reshape(np.asarray(x).shape)
actual_test=invert(y_test)
last_value_scaled=np.repeat(X_test[:,-1,0,None],HORIZON,axis=1)
seasonal_scaled=np.stack([X_test[:,LOOKBACK-7+(h%7),0] for h in range(HORIZON)],axis=1)
last_value=invert(last_value_scaled); seasonal=invert(seasonal_scaled)
def metrics(y,p):
    return {'MAE':mean_absolute_error(y.ravel(),p.ravel()),'RMSE':mean_squared_error(y.ravel(),p.ravel())**.5,'MAPE':np.mean(np.abs((y-p)/y))*100}
baseline_results=pd.DataFrame({'Last value':metrics(actual_test,last_value),'7-day seasonal':metrics(actual_test,seasonal)}).T
display(baseline_results.round(2))

                   MAE    RMSE   MAPE
Last value      146.62  197.13  11.23
7-day seasonal   53.18   93.39   3.96


## 5. TensorFlow LSTM model

In [6]:
tf.keras.backend.clear_session()
model=tf.keras.Sequential([
    tf.keras.layers.Input((LOOKBACK,1)),
    tf.keras.layers.Conv1D(32,5,padding='causal',activation='relu'),
    tf.keras.layers.LSTM(48,dropout=.15),
    tf.keras.layers.Dense(48,activation='relu'),
    tf.keras.layers.Dropout(.15),
    tf.keras.layers.Dense(HORIZON)
],name='energy_forecaster')
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),loss='mae',metrics=['mae'])
callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=6,min_delta=1e-4,restore_best_weights=True),tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss',patience=3,factor=.5,min_lr=1e-5)]
history=model.fit(X_train,y_train,validation_data=(X_val,y_val),epochs=40,batch_size=64,shuffle=False,callbacks=callbacks,verbose=0)
print({'epochs_trained':len(history.history['loss']),'best_validation_mae_scaled':min(history.history['val_loss'])})
pd.DataFrame(history.history)[['loss','val_loss']].plot(figsize=(9,4),title='Training history'); plt.ylabel('Scaled MAE'); plt.xlabel('Epoch'); plt.show()

{'epochs_trained': 40, 'best_validation_mae_scaled': 0.2646414339542389}


## 6. Untouched test evaluation

In [7]:
pred_test=invert(model.predict(X_test,verbose=0))
results=pd.DataFrame({'Last value':metrics(actual_test,last_value),'7-day seasonal':metrics(actual_test,seasonal),'TensorFlow LSTM':metrics(actual_test,pred_test)}).T
best_baseline=baseline_results.MAE.min(); improvement=1-results.loc['TensorFlow LSTM','MAE']/best_baseline
display(results.round(2)); print({'mae_improvement_vs_best_baseline_pct':round(improvement*100,2)})

horizon_rows=[]
for h in range(HORIZON):
    row=metrics(actual_test[:,h],pred_test[:,h]); row['horizon_day']=h+1; row['seasonal_MAE']=mean_absolute_error(actual_test[:,h],seasonal[:,h]); horizon_rows.append(row)
horizon_metrics=pd.DataFrame(horizon_rows)
fig,ax=plt.subplots(figsize=(10,4)); ax.plot(horizon_metrics.horizon_day,horizon_metrics.MAE,marker='o',label='LSTM'); ax.plot(horizon_metrics.horizon_day,horizon_metrics.seasonal_MAE,marker='o',label='Seasonal'); ax.set(xlabel='Forecast day',ylabel='MAE (GWh)',title='Error by forecast horizon'); ax.legend(); plt.show()
fig,ax=plt.subplots(figsize=(12,4)); ax.plot(pd.date_range(d_test[0],periods=HORIZON),actual_test[0],marker='o',label='Actual'); ax.plot(pd.date_range(d_test[0],periods=HORIZON),pred_test[0],marker='o',label='Forecast'); ax.plot(pd.date_range(d_test[0],periods=HORIZON),seasonal[0],ls='--',label='Seasonal baseline'); ax.legend(); ax.set(title='First untouched test forecast',ylabel='GWh'); plt.show()

                    MAE    RMSE   MAPE
Last value       146.62  197.13  11.23
7-day seasonal    53.18   93.39   3.96
TensorFlow LSTM   43.51   71.31   3.26
{'mae_improvement_vs_best_baseline_pct': np.float64(18.17)}


## 7. Calibrated uncertainty intervals

Absolute validation residuals calibrate a distribution-free interval for each horizon. This does not assume Gaussian errors. The test set is used only to report coverage and width—not to choose the interval.

In [8]:
pred_val=invert(model.predict(X_val,verbose=0)); actual_val=invert(y_val)
alpha=.10
q=np.quantile(np.abs(actual_val-pred_val),1-alpha,axis=0,method='higher')
lower=pred_test-q; upper=pred_test+q
coverage=np.mean((actual_test>=lower)&(actual_test<=upper)); avg_width=np.mean(upper-lower)
interval_metrics={'nominal_coverage':float(1-alpha),'empirical_test_coverage':float(coverage),'average_interval_width_gwh':float(avg_width)}
print(json.dumps(interval_metrics,indent=2))
fig,ax=plt.subplots(figsize=(12,4)); dates=pd.date_range(d_test[0],periods=HORIZON); ax.plot(dates,actual_test[0],marker='o',label='Actual'); ax.plot(dates,pred_test[0],marker='o',label='Forecast'); ax.fill_between(dates,lower[0],upper[0],alpha=.25,label='90% calibrated interval'); ax.legend(); ax.set(title='Forecast with validation-calibrated interval',ylabel='GWh'); plt.show()

{
  "nominal_coverage": 0.9,
  "empirical_test_coverage": 0.8837209302325582,
  "average_interval_width_gwh": 169.7863006591797
}


## 8. Operational error analysis

In [9]:
test_rows=pd.DataFrame({'forecast_start':pd.to_datetime(d_test),'actual_mean':actual_test.mean(1),'predicted_mean':pred_test.mean(1),'MAE':np.mean(np.abs(actual_test-pred_test),axis=1),'actual_peak':actual_test.max(1),'predicted_peak':pred_test.max(1)})
test_rows['season']=test_rows.forecast_start.dt.month.map({12:'Winter',1:'Winter',2:'Winter',3:'Spring',4:'Spring',5:'Spring',6:'Summer',7:'Summer',8:'Summer',9:'Autumn',10:'Autumn',11:'Autumn'})
display(test_rows.groupby('season').MAE.agg(['count','mean','median','max']).round(2))
display(test_rows.nlargest(10,'MAE'))
fig,ax=plt.subplots(figsize=(9,4)); sns.boxplot(data=test_rows,x='season',y='MAE',order=['Winter','Spring','Summer','Autumn'],ax=ax); ax.set_title('Forecast error by season'); plt.show()

        count       mean     median         max
season                                         
Autumn    182  48.299999  45.130001  170.080002
Spring    171  45.040001  44.419998   94.519997
Summer    184  24.440001  20.670000   71.769997
Winter    108  65.529999  53.279999  161.050003
    forecast_start  actual_mean  ...  predicted_peak  season
233     2016-11-02  1475.218018  ...     1409.120605  Autumn
644     2017-12-18  1321.803833  ...     1596.948364  Winter
292     2016-12-31  1482.141968  ...     1464.587280  Winter
643     2017-12-17  1335.382690  ...     1595.335693  Winter
598     2017-11-02  1457.651611  ...     1404.778198  Autumn
597     2017-11-01  1439.591064  ...     1410.823730  Autumn
642     2017-12-16  1347.341919  ...     1576.190918  Winter
294     2017-01-02  1506.403442  ...     1457.926147  Winter
291     2016-12-30  1459.438477  ...     1432.238159  Winter
281     2016-12-20  1318.408447  ...     1569.936768  Winter

[10 rows x 7 columns]


## 9. Save the model and verify inference

In [10]:
model_path=ROOT/'energy_demand_forecaster.keras'; model.save(model_path)
reloaded=tf.keras.models.load_model(model_path)
original=model.predict(X_test[:8],verbose=0); restored=reloaded.predict(X_test[:8],verbose=0)
max_delta=float(np.max(np.abs(original-restored))); assert max_delta<1e-6
print({'artifact':str(model_path),'size_mb':model_path.stat().st_size/1e6,'max_prediction_delta':max_delta})

{'artifact': 'energy_forecast/energy_demand_forecaster.keras', 'size_mb': 0.262159, 'max_prediction_delta': 0.0}


## 10. Acceptance tests, model card and CV evidence

In [11]:
checks={
 'real_dataset_rows':len(df)==4383,
 'continuous_daily_index':len(missing_dates)==0,
 'target_complete':df.consumption.notna().all(),
 'chronological_boundaries':d_train.max()<d_val.min()<d_test.min(),
 'train_only_scaler':np.isclose(scaler.mean_[0],values[:train_end,0].mean(),rtol=1e-5),
 'prediction_shape':pred_test.shape==actual_test.shape,
 'finite_predictions':np.isfinite(pred_test).all(),
 'probabilistic_order':np.all(lower<=pred_test) and np.all(pred_test<=upper),
 'model_beats_last_value':results.loc['TensorFlow LSTM','MAE']<results.loc['Last value','MAE'],
 'model_artifact_verified':model_path.exists() and max_delta<1e-6
}
display(pd.Series(checks,name='passed').to_frame()); assert all(checks.values())
run_summary={'test_start':str(pd.Timestamp(d_test.min()).date()),'test_end':str(df.index.max().date()),'test_windows':int(len(X_test)),'tensorflow':{k:float(v) for k,v in results.loc['TensorFlow LSTM'].items()},'seasonal_baseline':{k:float(v) for k,v in results.loc['7-day seasonal'].items()},'improvement_vs_best_baseline_pct':float(improvement*100),**interval_metrics}
print('RUN-DERIVED SUMMARY'); print(json.dumps(run_summary,indent=2))
print(f"CV bullet: Built a leakage-safe TensorFlow demand forecaster on {len(df):,} days of real German electricity data; achieved {results.loc['TensorFlow LSTM','MAE']:.1f} GWh test MAE ({improvement:.1%} improvement over the strongest baseline) with chronological validation, horizon diagnostics, calibrated intervals and verified model reload.")

model_card={'model':'Conv1D + LSTM multi-horizon forecaster','target':'next 14 days of German electricity consumption','intended_use':'portfolio demonstration and capacity-planning decision support','not_for':'automatic grid control or safety-critical dispatch','training_period':f"{df.index.min().date()} to {df.index[train_end-1].date()}",'metrics':run_summary,'limitations':['dataset ends in 2017 and may not represent current demand','univariate history omits weather, price, policy and structural changes','interval calibration can degrade under distribution shift','operational use requires rolling backtests, drift monitoring and retraining']}
print(json.dumps(model_card,indent=2))

                          passed
real_dataset_rows           True
continuous_daily_index      True
target_complete             True
chronological_boundaries    True
train_only_scaler           True
prediction_shape            True
finite_predictions          True
probabilistic_order         True
model_beats_last_value      True
model_artifact_verified     True
RUN-DERIVED SUMMARY
{
  "test_start": "2016-03-14",
  "test_end": "2017-12-31",
  "test_windows": 645,
  "tensorflow": {
    "MAE": 43.51486587524414,
    "RMSE": 71.30503288579109,
    "MAPE": 3.256746768951416
  },
  "seasonal_baseline": {
    "MAE": 53.17710494995117,
    "RMSE": 93.3877114423386,
    "MAPE": 3.9591517448425293
  },
  "improvement_vs_best_baseline_pct": 18.169923097169104,
  "nominal_coverage": 0.9,
  "empirical_test_coverage": 0.8837209302325582,
  "average_interval_width_gwh": 169.7863006591797
}
CV bullet: Built a leakage-safe TensorFlow demand forecaster on 4,383 days of real German electricity data; achie

## Conclusion

The neural forecast is accepted only if it beats simple operational baselines on the untouched future period. Horizon and seasonal diagnostics show where errors concentrate, while validation-calibrated intervals express planning uncertainty. Production work should add weather forecasts and calendar events, use rolling-origin backtesting, monitor residual drift and coverage, and retrain on current licensed data.